# Long Chau offline: isolated chunk delta
CPU only. No embedding, baseline overwrite or submission. Alias candidates require review.

In [ ]:
"""Source-text cleaning, applied before chunking. Every chunk row carries CLEAN_VERSION.

1. Line-break tags become line breaks: <br>, <br/>, <br />, &lt;br&gt;, and the fullwidth
   ＜br＞ / ＜br ／＞ found in about 1,077 zh documents. A run of tags (with the whitespace
   around it) becomes as many newlines as it had tags or existing newlines, capped at one blank
   line, so a break next to a paragraph break does not add a second blank line.
2. zh lead-in: a first paragraph starting with 核心提示 is dropped when the rest of the document
   repeats its text, because the site shows a summary before the article. A lead-in with no
   text is dropped too. A standalone summary that does not reappear is kept.

    核心提示：婴儿泪囊炎 婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病……\n\n婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病。一是可能……
    -> 婴儿泪囊炎眼睛有分泌物是一种眼睛轻度感染性疾病。一是可能……

Nothing else changes (no Unicode or whitespace normalisation).

    from text_clean import clean_text
"""
import re

CLEAN_VERSION = 'clean-v1'

BR_TAG = r'(?:<br\s*/?>|＜br\s*[/／]?\s*＞|&lt;br\s*/?&gt;)'
BR_RUN = re.compile(rf'[ \t　\n]*(?:{BR_TAG}[ \t　\n]*)+', re.IGNORECASE)
BR_ONE = re.compile(BR_TAG, re.IGNORECASE)
LEAD_IN = re.compile(r'^(?:核心提示[ \t　]*[:：][ \t　]*)+')
PARAGRAPH_BREAK = re.compile(r'\n\s*\n')


def line_break(match):
    run = match.group(0)
    return '\n' * min(max(len(BR_ONE.findall(run)), run.count('\n')), 2)


def drop_lead_in(text):
    """Drop a top 核心提示 paragraph whose first 40 characters (whitespace removed, trailing
    ellipsis removed) share a 10-character run with the rest of the document."""
    if not LEAD_IN.match(text):
        return text
    first, *rest = PARAGRAPH_BREAK.split(text, maxsplit=1)
    rest_text = rest[0] if rest else ''
    body = re.sub(r'\s+', '', LEAD_IN.sub('', first)).rstrip('….')
    if not body:
        return rest_text
    head, rest_plain = body[:40], re.sub(r'\s+', '', rest_text)
    if any(head[i:i + 10] in rest_plain for i in range(max(1, len(head) - 9))):
        return rest_text
    return text


def clean_text(text, lang):
    text = BR_RUN.sub(line_break, text)
    if lang == 'zh':
        text = drop_lead_in(text)
    return text


In [ ]:
"""Paragraph-first chunking with exact source offsets.

Every chunk is text[start:end] of the exported document, so chunk_text is always
a verbatim extract (SUBMISSION.md). Size is counted in words, or in characters/2
for Chinese. The scorer divides overlap by the *reference* chunk length, so a
chunk shorter than 40% of the reference can never match; err on the large side.

    uv run python scripts/chunking.py stats artifacts/corpus_v1/en.parquet --limits 120 250 400
"""
import argparse
import re
from dataclasses import dataclass

PARAGRAPH = re.compile(r'\n\s*\n')
SENTENCE_END = re.compile(r'(?<=[.!?。！？；;])\s+|(?<=[。！？；])')


@dataclass
class Chunk:
    order: int
    start: int
    end: int
    text: str


def size(text, lang):
    return len(text) // 2 if lang == 'zh' else len(text.split())


def spans(text, pattern, start=0, end=None):
    """Non-empty (start, end) spans of text[start:end] split by pattern, whitespace-trimmed."""
    end = len(text) if end is None else end
    cursor = start
    for match in list(pattern.finditer(text, start, end)) + [None]:
        stop = match.start() if match else end
        piece = text[cursor:stop]
        if piece.strip():
            left = cursor + len(piece) - len(piece.lstrip())
            right = cursor + len(piece.rstrip())
            yield left, right
        if match:
            cursor = match.end()


def sentences(text, start, end, lang, limit):
    """Sentence spans; a sentence above the limit (tables, lists without punctuation)
    is cut at line breaks, then hard-cut at whitespace (or every 2*limit chars for zh)."""
    for s_start, s_end in spans(text, SENTENCE_END, start, end):
        if size(text[s_start:s_end], lang) <= limit:
            yield s_start, s_end
            continue
        for l_start, l_end in spans(text, re.compile(r'\n'), s_start, s_end):
            if size(text[l_start:l_end], lang) <= limit:
                yield l_start, l_end
            elif lang == 'zh':
                for cut in range(l_start, l_end, 2 * limit):
                    yield cut, min(cut + 2 * limit, l_end)
            else:
                words = [m.span() for m in re.finditer(r'\S+', text[l_start:l_end])]
                for i in range(0, len(words), limit):
                    piece = words[i:i + limit]
                    yield l_start + piece[0][0], l_start + piece[-1][1]


def units(text, lang, limit):
    """Paragraphs, with paragraphs above the limit split into sentence groups."""
    for start, end in spans(text, PARAGRAPH):
        if size(text[start:end], lang) <= limit:
            yield start, end
            continue
        group = None
        for s_start, s_end in sentences(text, start, end, lang, limit):
            if group and size(text[group[0]:s_end], lang) > limit:
                yield group
                group = None
            group = (group[0], s_end) if group else (s_start, s_end)
        if group:
            yield group


def is_heading(text, lang):
    """Short line without closing punctuation: a section title or list label."""
    text = text.strip()
    short = len(text) <= 30 if lang == 'zh' else len(text.split()) <= 12
    return short and not text.endswith(('.', '!', '?', ':', ';', '。', '！', '？', '：', '；', '"', '”', ')'))


def chunk_document(text, lang, limit=350, minimum=40, overlap=50):
    """Greedily merge adjacent units up to `limit`; fold a tiny tail into the previous chunk.

    Headings never end a chunk: trailing headings move to the next chunk, so a title
    always stays with the text it introduces. A chunk that continues the same section
    also repeats the previous chunk's last whole sentences (up to `overlap` units);
    a chunk opening on a heading starts a new section and gets no overlap.
    """
    groups = []
    for unit in units(text, lang, limit):
        if groups and size(text[groups[-1][0][0]:unit[1]], lang) <= limit:
            groups[-1].append(unit)
            continue
        carried = []
        while groups and len(groups[-1]) > 1 and is_heading(text[groups[-1][-1][0]:groups[-1][-1][1]], lang):
            carried.insert(0, groups[-1].pop())
        groups.append(carried + [unit])
    merged = [(group[0][0], group[-1][1]) for group in groups]
    if len(merged) > 1 and size(text[merged[-1][0]:merged[-1][1]], lang) < minimum:
        tail = merged.pop()
        merged[-1] = (merged[-1][0], tail[1])
    chunks = []
    for order, (start, end) in enumerate(merged):
        if order and overlap and not is_heading(text[start:end].split('\n', 1)[0], lang):
            previous_start, previous_end = merged[order - 1]
            for s_start, _ in reversed(list(spans(text, SENTENCE_END, previous_start, previous_end))):
                if size(text[s_start:previous_end], lang) > overlap:
                    break
                start = s_start
        chunks.append(Chunk(order, start, end, text[start:end]))
    return chunks


def sample_rows(path, sample):
    """(lang, text) of a random sample, read one row group at a time: take() on the whole
    text column overflows the 2 GB string offsets of the 1 GB vi corpus."""
    import numpy as np
    import pyarrow.parquet as pq

    file = pq.ParquetFile(path)
    total = file.metadata.num_rows
    wanted = None
    if sample and total > sample:
        wanted = set(np.random.default_rng(0).choice(total, sample, replace=False).tolist())
    rows, offset = [], 0
    for group in range(file.metadata.num_row_groups):
        count = file.metadata.row_group(group).num_rows
        columns = file.read_row_group(group, columns=['lang', 'text']).to_pydict()
        rows += [(columns['lang'][i], columns['text'][i]) for i in range(count)
                 if wanted is None or offset + i in wanted]
        offset += count
    return rows


def stats(path, limits, sample):
    import numpy as np

    rows = sample_rows(path, sample)
    langs, texts = [lang for lang, _ in rows], [text for _, text in rows]
    lengths = np.array([size(t, l) for t, l in zip(texts, langs)])
    unit = 'chars/2' if langs and langs[0] == 'zh' else 'words'
    print(f'{path}: {len(texts):,} documents sampled, length in {unit}')
    print('  document length p10/p50/p90/p99:', *np.percentile(lengths, [10, 50, 90, 99]).astype(int))
    for limit in limits:
        counts, chunk_sizes = [], []
        for text, lang in zip(texts, langs):
            chunks = chunk_document(text, lang, limit)
            counts.append(len(chunks))
            chunk_sizes.extend(size(c.text, lang) for c in chunks)
            assert all(text[c.start:c.end] == c.text for c in chunks)
        scale = len(texts) if not sample else None
        print(f'  limit {limit:>4}: chunks/doc mean {np.mean(counts):.2f}, '
              f'chunk size p50/p90 {int(np.percentile(chunk_sizes, 50))}/{int(np.percentile(chunk_sizes, 90))}, '
              f'single-chunk docs {np.mean(np.array(counts) == 1):.0%}'
              + (f', total {sum(counts):,}' if scale else ''))


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument('action', choices=['stats'])
    parser.add_argument('parquet')
    parser.add_argument('--limits', type=int, nargs='+', default=[120, 250, 400])
    parser.add_argument('--sample', type=int, default=20000, help='0 = all documents')
    args = parser.parse_args()
    stats(args.parquet, args.limits, args.sample)


if False:
    main()


In [ ]:
PAYLOAD_B64 = '...'
EXPECTED_PAYLOAD_SHA = '71993b89e8b22d44c2618f1fc97663a351c287ee33e239578843ec653d2bb580'
BATCH_MANIFEST = {'format_version': 1, 'batch_id': 'longchau-offline-20261029', 'source_sqlite_sha256': '992ab6875c46c30af4253bd5db22244eb43171e0cb6841be175ef044bd19755b', 'official_sha256': '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7', 'baseline_version': 'v1', 'baseline_built_at': '2026-10-21T15:56:12+00:00', 'baseline_registry_sha256': 'a2472d74765e1554b84d9683c526bfef847404b2ac16f6be76bcb69ab79f5377', 'parser_version': 'parser-v2.5', 'clean_version': 'clean-v1', 'source_html_rows': 285, 'already_in_baseline': 166, 'candidates': 119, 'accepted': 116, 'review': 3, 'raw_text_matches_baseline': 65, 'chunk_params': {'max_size': 350, 'overlap': 50, 'min_size': 40}, 'chunks_prepared': False, 'embeddings_prepared': False, 'files': {'documents.jsonl.gz': {'bytes': 277819, 'sha256': '71993b89e8b22d44c2618f1fc97663a351c287ee33e239578843ec653d2bb580'}, 'documents.parquet': {'bytes': 313849, 'sha256': '90f6ed04edef035b8316eb4490a77acab3b7f33b9b35237df326d876ac8f5b49'}, 'needs_review.json': {'bytes': 486, 'sha256': '606c43f0013af4b5cbd0f2701d782955d0380b275a1b2d11c1b45704818a077d'}, 'new_text_documents.jsonl.gz': {'bytes': 119324, 'sha256': 'e04cf0d60dd25b424676adfde14430799575b02f00517bfb5e4eab0a8120cf26'}, 'alias_candidate_documents.jsonl.gz': {'bytes': 159543, 'sha256': '29daef953c31aab253b7a98b28644845b602d00531f98314086b8e14890b3eef'}}, 'limitations': ['Compared against archive.zip corpus v1 only; recheck later patches before merging.', 'Baseline raw text hashes are not a substitute for chunk/vector hash comparison.', 'No baseline, checkpoint, chunks, vectors or leaderboard submissions changed.'], 'new_raw_text_docs': 51, 'unique_new_raw_texts': 51, 'alias_candidate_docs': 65}


In [ ]:

import base64, gzip, hashlib, inspect, json
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

payload=base64.b64decode(PAYLOAD_B64)
assert hashlib.sha256(payload).hexdigest()==EXPECTED_PAYLOAD_SHA
records=[json.loads(s) for s in gzip.decompress(payload).decode().splitlines()]
params=inspect.signature(chunk_document).parameters
chunk_params='/'.join(str(params[n].default) for n in ['limit','overlap','minimum'])
assert chunk_params=='350/50/40' and CLEAN_VERSION==BATCH_MANIFEST['clean_version']

# Attach ntklinhfitus/medxir-chunk-corpus; full corpus registry is optional.
inputs=Path('/kaggle/input')
registries=list(inputs.rglob('registry.parquet'))
manifests=list(inputs.rglob('chunk_manifest.json'))
assert len(registries)<=1 and len(manifests)==1,(registries,manifests)
registry_path=registries[0] if registries else None
chunk_manifest_path=manifests[0]
if registry_path is not None:
    assert hashlib.sha256(registry_path.read_bytes()).hexdigest()==BATCH_MANIFEST['baseline_registry_sha256'], 'Baseline changed: rebuild data audit first'
base_manifest=json.loads(chunk_manifest_path.read_text())
assert base_manifest['chunk_params']==chunk_params
assert base_manifest['clean_version']==CLEAN_VERSION
assert base_manifest['dataset_version']==BATCH_MANIFEST['baseline_version']
assert base_manifest['dataset_built_at']==BATCH_MANIFEST['baseline_built_at']
groups_path=chunk_manifest_path.parent/'hash_groups.parquet'
assert groups_path.is_file()
groups=pd.read_parquet(groups_path)
by_sha={r.text_sha256:r for r in groups.itertuples(index=False)}
reg=None
if registry_path is not None:
    registry=pd.read_parquet(registry_path,columns=['doc_id','url','state'])
    reg=registry.set_index('doc_id')
else:
    print('No full registry attached; use packaged ID/URL audit and matching baseline chunk manifest.')

out=Path('/kaggle/working/data_delta_20261029');out.mkdir(exist_ok=True)
new_chunks=[];alias_rows=[];new_groups=[];todo=[];seen_ids=set();seen_hashes=set()
for d in records:
    assert d['doc_id'] not in seen_ids;seen_ids.add(d['doc_id'])
    if reg is not None:
        assert d['doc_id'] in reg.index and reg.loc[d['doc_id'],'url']==d['url']
        assert reg.loc[d['doc_id'],'state']!='in_corpus'
    else:
        assert d['baseline_state']=='pending'
        assert d['url'].startswith('https://nhathuoclongchau.com.vn/')
    sha=hashlib.sha256(d['text'].encode()).hexdigest()
    assert sha==d['text_sha256']
    if sha in by_sha:
        old=by_sha[sha]
        alias_rows.append(dict(doc_id=d['doc_id'],text_sha256=sha,
            baseline_rep_doc_id=int(old.rep_doc_id),baseline_fanout_safe=bool(old.fanout_safe),
            baseline_group_type=old.group_type,decision='manual_mapping_review'))
        continue
    if sha in seen_hashes:
        alias_rows.append(dict(doc_id=d['doc_id'],text_sha256=sha,decision='duplicate_within_delta_review'))
        continue
    seen_hashes.add(sha)
    text=clean_text(d['text'],d['lang'])
    for c in chunk_document(text,d['lang']):
        assert text[c.start:c.end]==c.text
        cid=f'{sha[:16]}-{c.order}'
        row=dict(chunk_id=cid,text_sha256=sha,rep_doc_id=d['doc_id'],lang=d['lang'],
            order=c.order,start=c.start,end=c.end,chunk_text=c.text,n_units=size(c.text,d['lang']),
            title=d['title'],clean_version=CLEAN_VERSION,chunk_params=chunk_params)
        new_chunks.append(row)
        todo.append(dict(chunk_id=cid,text_sha256=sha,
            chunk_text_sha256=hashlib.sha256(c.text.encode()).hexdigest(),
            embedding_state='needs_per_model_vector_inventory'))
    new_groups.append(dict(text_sha256=sha,rep_doc_id=d['doc_id'],doc_ids=[d['doc_id']],
                          n_ids=1,group_type='single',fanout_safe=True))
assert len({r['chunk_id'] for r in new_chunks})==len(new_chunks)
# No mutation of existing hash groups; new IDs on old text are review-only.
for name,rows in [('new_chunks',new_chunks),('new_hash_groups',new_groups),
                  ('alias_candidates',alias_rows),('embedding_todo',todo)]:
    if rows: pq.write_table(pa.Table.from_pylist(rows),out/(name+'.parquet'),compression='zstd')
    (out/(name+'.json')).write_text(json.dumps(rows,ensure_ascii=False),encoding='utf-8')
summary=dict(batch_id=BATCH_MANIFEST['batch_id'],accepted_docs=len(records),
    new_text_groups=len(new_groups),alias_candidates=len(alias_rows),new_chunks=len(new_chunks),
    baseline_manifest=base_manifest,clean_version=CLEAN_VERSION,chunk_params=chunk_params,
    baseline_modified=False,vectors_generated=False,full_registry_rechecked=reg is not None,
    limitations=['Recheck patches newer than corpus v1 before merge.',
        'Alias candidates are not automatically added to fanout.',
        'Each model must verify input/config compatibility before vector reuse.'])
summary['files']={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in out.iterdir() if p.is_file()}
(out/'delta_chunk_manifest.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if k not in ['baseline_manifest','files']},indent=2))
